In [21]:
!uv pip install magika aiofile langgrap langchain langchain-core langchain_openai langchain_text_splitters tree-sitter-python tree-sitter-language-pack python-frontmatter &> /dev/null

In [2]:
import pathlib as plb
import subprocess
from collections.abc import Generator
from functools import cached_property

import frontmatter as fm
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter
from magika import Magika
from pydantic import BaseModel
from tree_sitter_language_pack import DownloadError, get_parser

In [3]:
path = plb.Path(".").resolve().parent.parent

## File readers

In [4]:
class FileMetadataModel(BaseModel):
    lang_name: str
    description: str
    is_text: bool
    score: float
    

class FileWithMedatadataModel(BaseModel):
    content: bytes
    metadata: FileMetadataModel
    path: plb.Path
    
    @cached_property
    def content_str(self):
        return self.content.decode('utf-8') 

In [5]:
class FileReader:
    
    def __init__(self):
        self._language_predictor = Magika()
    
    @staticmethod
    def _read_bytes(file_path: plb.Path) -> bytes:
        return file_path.read_bytes()
            
    def _get_file_metadata(self, content: bytes) -> FileMetadataModel:
        metadata = self._language_predictor.identify_bytes(content)
        return FileMetadataModel(
            lang_name=metadata.output.label,
            description=metadata.output.description,
            is_text=metadata.output.is_text,
            score=metadata.score
        )
        
    def read_file(self, file_path: plb.Path) -> FileWithMedatadataModel:
        content = self._read_bytes(file_path)
        metadata = self._get_file_metadata(content)
        return FileWithMedatadataModel(content=content, metadata=metadata, path=file_path)
        
    def read_files(self, file_paths: Generator[plb.Path, None, None]) -> Generator[FileWithMedatadataModel, None, None]:
        for file_path in file_paths:
            yield self.read_file(file_path)

In [6]:
class ExclusionFileReader:
    
    def __init__(self, reader: FileReader, excluded_directories: list[str], included_extensions: list[str]):
        self._reader = reader
        self._excluded_directories = set(excluded_directories)
        self._included_extensions = set(included_extensions)
        
    def _create_git_cmd(self, path: plb.Path) -> list[str]:
        cmd = ["git", "-C", str(path), "ls-files", "--cached", "--others", "--exclude-standard"]
        cmd.append("--")
        cmd.append(".")
        for dir in self._excluded_directories:
            cmd.append(f":(exclude,glob)**/{dir}/**")
        return cmd
    
    def _run_git_cmd(self, cmd: list[str]) -> subprocess.CompletedProcess[str]:
        return subprocess.run(
            cmd,
            capture_output=True, text=True, check=True
        )
        
    def _parse_git_cmd_stdout(self, subprocess_result: subprocess.CompletedProcess[str]) -> list[str]:
        return subprocess_result.stdout.splitlines()
    
    def _filter_file_paths(self, root_path: plb.Path, file_paths: list[str]) -> Generator[plb.Path, None, None]:
        for line in file_paths:
            path = root_path / line
            if path.suffix in self._included_extensions and path.is_file():
                yield path
        
    def read_files(self, path: plb.Path) -> Generator[FileWithMedatadataModel, None, None]:
        file_paths_from_git = self._parse_git_cmd_stdout(
            self._run_git_cmd(
                self._create_git_cmd(
                    path=path
                )
            )
        )
        file_paths_filtered = self._filter_file_paths(root_path=path, file_paths=file_paths_from_git)
        return self._reader.read_files(file_paths_filtered)

Exclusion file reader ignores any paths also ignored by git, and user can then additionally ignore folders they don't need parsed. Included extensions should be set explicitly.

In [7]:
_reader = FileReader()
included_extensions=[".py", ".toml", ".yaml", ".yml", ".jinja", ".sh", ".bash", ".md", ".json"]
excluded_directories=[".loop", ".claude", ".vscode", ".okf", "docs", ".claude"]
root_path=plb.Path("/Users/jasperginn/Documents/code_projects/kmctl")

reader = ExclusionFileReader(reader=_reader, excluded_directories=excluded_directories, included_extensions=included_extensions)

files = [f for f in reader.read_files(root_path)]

We can retrieve required parsers and re-use them instead of setting up new ones for each parsing operation.

In [8]:
extensions = list(set([f.metadata.lang_name for f in files]))

parsers = {}
for ext in extensions:
    try:
        parsers[ext] = get_parser(ext)
    except DownloadError:
        print(f"No parser for {ext}")

No parser for jinja
No parser for txt


In [9]:
parsers

{'json': <tree_sitter.Parser at 0x1113fbe10>,
 'yaml': <tree_sitter.Parser at 0x113fd44e0>,
 'python': <tree_sitter.Parser at 0x113fd4a80>,
 'toml': <tree_sitter.Parser at 0x113fd43c0>,
 'markdown': <tree_sitter.Parser at 0x113fd5620>,
 'shell': <tree_sitter.Parser at 0x113fd4960>}

Parsing files then becomes quite straightforward

In [10]:
files_parsed = []
for f in files:
    lang_name = f.metadata.lang_name
    if lang_name not in parsers:
        continue
    else:
        _parser = parsers[lang_name]
        files_parsed.append(_parser.parse(f.content))

In [11]:
len(files_parsed)

138

## Chunking markdown

To extract claims from markdown, ideally we'd like to extract stable chunks. This allows us to leave unchanged chunks alone.

In [167]:
class MarkdownChunker:
    
    def __init__(self, skip: list[str] = ["index.md", "log.md"]):
        self._headers = [
            ("---", 'Frontmatter'),
            ("#", 'Header 1'),
            ('##', 'Header 2'),
            ('###', 'Header 3')
        ]
        self._splitter = MarkdownHeaderTextSplitter(self._headers)
        self._reader = FileReader()
        self._skip = skip
        
    @staticmethod
    def _split_frontmatter(content: str) -> tuple[dict[str, object], str]:
        _content = fm.loads(content)
        return _content.metadata, _content.content
        
    def _chunk(self, content: str) -> list[Document]:
        return self._splitter.split_text(content)
        
    def parse(self, path: plb.Path, glob: str = "**/*.md") -> Generator[list[Document], None, None]:
        for f in self._reader.read_files(plb.Path(path).glob(glob)):
            if f.path.name in self._skip:
                continue
            if f.metadata.lang_name != "markdown":
                continue
            metadata, content_str = self._split_frontmatter(f.content_str)
            if metadata.get("status", "unknown") == "deprecated":
                continue
            yield self._chunk(content_str)

In [168]:
chunker = MarkdownChunker()

okf_chunked = [*chunker.parse(path=plb.Path("/Users/jasperginn/Documents/code_projects/kmctl/.okf"))]

In [169]:
len(okf_chunked)

30

## Extract claims from the markdown docs

In [170]:
import operator
import os
from typing import Annotated

from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessagesState, StateGraph
from typing_extensions import TypedDict

In [171]:
llm = ChatOpenAI(
    base_url="https://bifrost.lab.orangecluster.nl/v1",
    model="gemini/gemini-3.5-flash-lite",
    #model="ollama/deepseek-v4.1-flash",
    api_key=os.environ["BIFROST_CC_VIRTUAL_KEY"] # type: ignore
)

In [172]:
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls: int

In [173]:
msg = """
Your job is to extract claims from the provided markdown texts.

In this context, a **claim** is a statement about the code that should be
implemented in the project's code base, or provides an abstract rule that
should be followed.

The following fields are mandatory:
- claim (str): the claim, reworded in at most three sentences.
- evidence (array of str): what evidence underpins the claim, as quotes from the original
text
- claim_type (str): one of 
 * **point**: points to a specific function, class, config etc.
 * **structural**: wide claims, rules about e.g. imports, modules, 'all of X'
 * **subjective**: claims that describe a vision or architectural principles that are unverifiable from code (e.g. 'all code needs to be decoupled').
- links (array of str): any links or paths mentioned relevant to the claim

Return only valid JSON. Do not wrap it in backticks.
""".strip()

system_message = SystemMessage(
    content=msg
)

In [174]:
def llm_call(state: MessagesState) -> MessagesState:
    ai = llm.invoke([system_message] + state["messages"])
    return {"messages": [ai], "llm_calls": state.get("llm_calls", 0) + 1}

In [175]:
agent_builder = StateGraph(MessagesState)
agent_builder.add_node("llm_call", llm_call)

agent_builder.add_edge(START, "llm_call")
agent_builder.add_edge("llm_call", END)

agent = agent_builder.compile()


In [176]:
chunker = MarkdownChunker()

okf_chunked = chunker.parse(path=plb.Path("/Users/jasperginn/Documents/code_projects/kmctl/.okf"))

chnks = [*okf_chunked]

In [177]:
len(chnks)

30

In [181]:
import json

messages = []

for chnk in chnks[25]:
    msg = f"""
    Extract claims from the following markdown chunk:

    ```markdown
    {chnk.page_content}
    ```
    """.strip()

    prompt = {
        "messages": [
            HumanMessage(msg)
        ],
        "llm_calls": 0
    }
    
    messages.append(prompt)

result = await agent.abatch(
    inputs = messages
)

In [184]:
[r["messages"][-1].content for r in result]

['[\n  {\n    "claim": "okf-skills provides a Stop hook named okf-stop-check.sh that prevents an agent from stopping if tracked files have changed while nothing under .okf/ was modified.",\n    "evidence": [\n      "okf-skills ships a Stop hook, `okf-stop-check.sh`, that keeps an agent from stopping when tracked files changed and nothing under `.okf/` did."\n    ],\n    "claim_type": "point",\n    "links": [\n      "0015-antigravity-gets-skills-and-rules-from-apm-and-a-kmctl-plugin-folder.md"\n    ]\n  },\n  {\n    "claim": "The okf-stop-check.sh hook remains inactive unless the bundle explicitly opts in by setting upkeep: enforced in .okf/index.md.",\n    "evidence": [\n      "It does nothing unless the bundle opts in with `upkeep: enforced` in `.okf/index.md`."\n    ],\n    "claim_type": "point",\n    "links": [\n      "0015-antigravity-gets-skills-and-rules-from-apm-and-a-kmctl-plugin-folder.md"\n    ]\n  }\n]',
 '[\n  {\n    "claim": "kmctl writes a Stop hook for okf\'s check in An

In [183]:
jsons = [json.loads(r["messages"][-1].content) for r in result]

jsons

JSONDecodeError: Expecting ',' delimiter: line 13 column 66 (char 626)

In [137]:
jsons

[[{'claim': 'kmctl puts the right knowledge in front of the agent at the start of every session, including where knowledge lives, rules for writing it, and what is due for review.',
   'evidence': ['1. **It puts the right knowledge in front of the agent** at the start of every session: where knowledge lives, the rules for writing it, and what is due for review. Injecting into context works in every harness; a page the agent has to find does not (ADR 0002).'],
   'claim_type': 'structural',
   'links': ['ADR 0002']},
  {'claim': 'kmctl provides an auditing function where a separate agent checks on request or in CI that decisions hold, architecture matches code, and practices have evidence.',
   'evidence': ['2. **It audits.** A separate agent checks, on request or in CI, that decisions still hold, the architecture still matches the code and the practices still have their evidence. It reports; a person decides\n(ADR 0005).'],
   'claim_type': 'structural',
   'links': ['ADR 0005']},
  {'

In [ ]:
OKF_PATH = '/Users/jasperginn/Documents/code_projects/kmctl/.okf'

reader = FileReader()

files = [*reader.read_files(plb.Path(OKF_PATH).glob("**/*.md"))]

In [229]:
len(files)

44

In [230]:
files = [f for f in files if f.metadata.lang_name == "markdown"]

# NB: three markdown files contain plain text content

len(files)

41

In [232]:
split = splitter.split_text(files[0].content_str)

In [235]:
split

[Document(metadata={'Frontmatter': ''}, page_content='type: summary\ntitle: "Summary: kmctl"\ndescription: "The whole of kmctl in at most 500 words: what it is for, how its parts fit, what its records settle and where it stands, citing the concepts; start here."\ngenerated:\nby: claude-opus/5.5\nat: 2026-10-18\ntags: [summary]\nchecked:\nat: 2026-10-20T09:09:13Z\nagainst:\n- architecture/overview\n- decisions/0035-kmctl-s-repository-answers-and-triages-with-hermes'),
 Document(metadata={'Frontmatter': '', 'Header 1': 'Summary: kmctl', 'Header 2': 'The project'}, page_content="kmctl keeps a repository's knowledge current: `docs/` for people who use a\nproject, an `.okf/` bundle for people and agents working on it. Since\nnothing can force knowledge to stay true, kmctl puts it in front of the\nagent at every session start and audits it; a person decides\n([manifesto](manifesto.md)). It serves maintainers and their agents in\nClaude Code, Antigravity and OpenCode."),
 Document(metadata={'

In [82]:
parser = get_parser("hcl")

In [83]:
CODE_FP = "/Users/jasperginn/Documents/clients/rituals/RAIL/repositories/rail-agentic-templates/terraform/archetypes/cloud-run-agent/cloud_sql.tf"

In [84]:
b = plb.Path(CODE_FP).read_bytes()

In [85]:
tree = parser.parse(b)

In [198]:
def print_tree(node, indent=0):
    # Print the node type and its line numbers
    print("  " * indent + f"{node.type} [{node.start_point[0]}:{node.start_point[1]}]")
    
    # If it's a leaf node (has no children), print the actual code text
    if len(node.children) == 0:
        print("  " * (indent + 1) + f"-> {node.text.decode('utf-8')}")
        
    for child in node.children:
        print_tree(child, indent + 1)

In [95]:
print_tree(tree.root_node)

config_file [0:0]
  body [0:0]
    block [0:0]
      identifier [0:0]
        -> locals
      block_start [0:7]
        { [0:7]
          -> {
      comment [1:2]
        -> # One DSN for both ADK's DatabaseSessionService and the A2A DatabaseTaskStore, over the
      comment [2:2]
        -> # managed proxy socket that psycopg3's `host=<dir>` selects. The driver MUST be
      comment [3:2]
        -> # `+psycopg`: both stores use create_async_engine, which rejects the psycopg2 dialect.
      body [4:2]
        attribute [4:2]
          identifier [4:2]
            -> db_instance_name
          = [4:25]
            -> =
          expression [4:27]
            conditional [4:27]
              expression [4:27]
                variable_expr [4:27]
                  identifier [4:27]
                    -> var
                get_attr [4:30]
                  . [4:30]
                    -> .
                  identifier [4:31]
                    -> create_db_instance
              ? [4:5

In [107]:
m = Magika()

mr = m.identify_bytes(b)

label = mr.prediction.dl.label

In [ ]:
parser = get_parser(label)

In [111]:
mr

MagikaResult(path=-, status=ok, prediction=MagikaPrediction(dl=ContentTypeInfo(label=hcl, mime_type='text/x-hcl', group='code', description='HashiCorp configuration language', extensions=['hcl'], is_text=True), output=ContentTypeInfo(label=hcl, mime_type='text/x-hcl', group='code', description='HashiCorp configuration language', extensions=['hcl'], is_text=True), score=0.9997134804725647, overwrite_reason=<OverwriteReason.NONE: 'none'>))